In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [2]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_32230/4196082941.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_32230/4196082941.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
    def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q_val),
                0, 1, n=n_points
            )[0]

    integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
    
    return integral_value

In [6]:
# def model function
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [7]:
# set cost and minimize
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.93 (χ²/ndof = 0.2)      │              Nfcn = 341              │
│ EDM = 3.1e-06 (Goal: 0.0002)     │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0616   │  0.0022   │            │            │         │         │       │
│ 1 │ mg   │   0.389   │   0.005   │            │            │         │         │       │
│ 2 │ a1   │   1.49    │   0.05    │            │            │         │         │       │
│ 3 │ a2   │   2.16    │   0.31    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 4.85e-06    10e-6    54e-6  -157e-6 │
│  mg │    10e-6 2.41e-05 0.056e-3 0.101e-3 │
│  a1 │    54e-6 0.056e-3   0.0023  -0.0136 │
│  a2 │  -157e-6 0.101e-3  -0.0136   0.0956 │
└─────┴─────────────────────────────────────┘

In [8]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001
    n_points = 10000

    # def integrand(y, x, mg, a1, a2, m2_func, q_val):
    #     k        = sqrt_s * x
    #     phi      = 2*np.pi*y
    #     jacobian = 2*np.pi*sqrt_s
    #     return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
    #               T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        # def inner_integral(x):
        #     return fixed_quad(
        #         lambda y: integrand(y, x, mg, a1, a2, mg_model, q2),
        #         0, 1, n=n_points
        #     )[0]

        # integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]

        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)


        diff_T = integral_value
        # print(f"q2: {q2}, diff_T: {diff_T}")

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#-----------------------------------------------------------------------------------------------

#-----------------------------------------------------------------------------------------------

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': 'blue', 'width': 2},
              'marker': {'size': 4},
              'mode': 'lines+markers',
              'name': '7 TeV, pl',
              'showlegend': True,
              'type': 'scatter',
              'x': [0.006, 0.007, 0.008, 0.009000000000000001,
                    0.010000000000000002, 0.011000000000000003,
                    0.012000000000000004, 0.013000000000000005,
                    0.014000000000000005, 0.015000000000000006,
                    0.016000000000000007, 0.017000000000000008,
                    0.01800000000000001, 0.01900000000000001, 0.02000000000000001,
                    0.02100000000000001, 0.022000000000000013,
                    0.023000000000000013, 0.024000000000000014,
                    0.025000000000000015, 0.026000000000000016,
                    0.027000000000000017, 0.028000000000000018,
                    0.02900000000000002, 0.03000000000000002, 0.03100000000000002,
                    0.03200000000000002, 0.03300000000000002, 0.03400000000000002,
                    0.035000000000000024, 0.036000000000000025,
                    0.037000000000000026, 0.03800000000000003, 0.03900000000000003,
                    0.04000000000000003, 0.04100000000000003, 0.04200000000000003,
                    0.04300000000000003, 0.04400000000000003, 0.04500000000000003,
                    0.046000000000000034, 0.047000000000000035,
                    0.048000000000000036, 0.04900000000000004, 0.05000000000000004,
                    0.05100000000000004, 0.05200000000000004, 0.05300000000000004,
                    0.05400000000000004, 0.05500000000000004, 0.05600000000000004,
                    0.057000000000000044, 0.058000000000000045,
                    0.059000000000000045, 0.060000000000000046,
                    0.06100000000000005, 0.06200000000000005, 0.06300000000000004,
                    0.06400000000000004, 0.06500000000000004, 0.06600000000000004,
                    0.06700000000000005, 0.06800000000000005, 0.06900000000000005,
                    0.07000000000000005, 0.07100000000000005, 0.07200000000000005,
                    0.07300000000000005, 0.07400000000000005, 0.07500000000000005,
                    0.07600000000000005, 0.07700000000000005, 0.07800000000000006,
                    0.07900000000000006, 0.08000000000000006, 0.08100000000000006,
                    0.08200000000000006, 0.08300000000000006, 0.08400000000000006,
                    0.08500000000000006, 0.08600000000000006, 0.08700000000000006,
                    0.08800000000000006, 0.08900000000000007, 0.09000000000000007,
                    0.09100000000000007, 0.09200000000000007, 0.09300000000000007,
                    0.09400000000000007, 0.09500000000000007, 0.09600000000000007,
                    0.09700000000000007, 0.09800000000000007, 0.09900000000000007,
                    0.10000000000000007, 0.10100000000000008, 0.10200000000000008,
                    0.10300000000000008, 0.10400000000000008, 0.10500000000000008,
                    0.10600000000000008, 0.10700000000000008, 0.10800000000000008,
                    0.10900000000000008, 0.11000000000000008, 0.11100000000000008,
                    0.11200000000000009, 0.11300000000000009, 0.11400000000000009,
                    0.11500000000000009, 0.11600000000000009, 0.11700000000000009,
                    0.11800000000000009, 0.11900000000000009, 0.12000000000000009,
                    0.1210000000000001, 0.1220000000000001, 0.1230000000000001,
                    0.1240000000000001, 0.12500000000000008, 0.12600000000000008,
                    0.12700000000000009, 0.12800000000000009, 0.1290000000000001,
                    0.1300000000000001, 0.1310000000000001, 0.1320000000000001,
                    0.1330000000000001, 0.1340000000000001, 0.1350000000000001,
                    0.1360000000000001, 0.1370000000000001, 0.1380000000000001,
                    0.139000000000000

In [17]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:
        s = sqrt_s ** 2


        integral_value = full_int(mg, a1, a2, mg_model, 0.0, sqrt_s) 

        born_amp = amp_calculation(integral_value, s, epsilon, 0)
        lst_born_amp.append(born_amp)
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon, 0), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    minuit_born.values['eps'],
    minuit_born.values['mg'],
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

fig.show(renderer="browser")

/tmp/ipykernel_32230/2960580461.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [10]:
def eik_amp(s, b, q, chi):
    return 1j * s * b * j0(b*q) * (1 - np.exp(1j*chi))

print(lst_born_amp)

[78.41389592681145j, 1490109.4873411304j, 6423638.869562708j, 15139899.987069093j, 27837066.141512733j, 44660155.46798415j, 65724752.28819218j, 91127489.3288743j, 120951688.08989424j, 155270767.32100576j, 194150469.84636205j, 237650401.90699938j, 285825143.95655876j, 338725079.7989731j, 396397032.6674698j, 458884764.3689249j, 526229374.5007638j, 598469624.9733326j, 675642207.5383229j, 757781967.0470047j, 844922089.7807616j, 937094263.8426802j, 1034328816.9251059j, 1136654835.5549896j, 1244100269.0246596j, 1356692020.546476j, 1474456027.6620028j, 1597417333.5462039j, 1725600150.5439067j, 1859027917.0376782j, 1997723348.5574195j, 2141708483.8909173j, 2291004726.8328023j, 2445632884.1102986j, 2605613199.943152j, 2770965387.628281j, 2941708658.4844136j, 3117861748.445721j, 3299442942.5547743j, 3486470097.5725217j, 3678960662.895318j, 3876931699.9455247j, 4080399900.1821365j, 4289381601.86055j, 4503892805.65588j, 4723949189.251216j, 4949566120.981221j, 5180758672.611655j, 5417541631.327026j

In [28]:
import numpy as np
from scipy.special import j0

# --- setup parameters ---
step = 200
lst_amp_eik = []

lst_q_integration = np.linspace(0, 0.2, 50)
lst_b_integration = np.linspace(0, 30, 50)
# lst_sqrt_s = np.arange(500, 13000, step)
lst_sqrt_s_eik = np.linspace(500, 13000, 131)

# compute step sizes for the Riemann sum
dq = lst_q_integration[1] - lst_q_integration[0]
db = lst_b_integration[1] - lst_b_integration[0]

for sqrt_s in lst_sqrt_s_eik:
    s = sqrt_s**2

    # initialize amplitude for this sqrt(s)
    eik_amp_sum = 0

    # --- integration over b ---
    for b_val in lst_b_integration:
        chi_sum = 0

        # --- integration over q ---
        for q_val in lst_q_integration:
            # full_int must be your differential amplitude function
            diff_t = full_int(
                ensemble_parameters['atlas']['pl']['mg'],
                ensemble_parameters['atlas']['pl']['a1'],
                ensemble_parameters['atlas']['pl']['a2'],
                m2_pl,
                q_val,
                sqrt_s
            )

            t = -q_val
            born_amp = amp_calculation(
                diff_t,
                s,
                ensemble_parameters['atlas']['pl']['epsilon'],
                t
            )

            chi_val = (1 / s) * (q_val * j0(b_val * q_val)) * born_amp

            # Riemann sum contribution (Δq)
            chi_sum += chi_val * dq

        # after q integration, apply factor and accumulate over b (Δb)
        factor = 1 - np.exp(1j * chi_sum)
        eik_amp_value = (1j * s) * b_val * j0(b_val * 0) * factor
        eik_amp_sum += eik_amp_value * db

    lst_amp_eik.append(eik_amp_sum)
    print(f"√s = {sqrt_s:.0f} GeV -> Eikonal amplitude = {eik_amp_sum}")



√s = 500 GeV -> Eikonal amplitude = 21456947.609921902j
√s = 596 GeV -> Eikonal amplitude = 31120034.390653092j
√s = 692 GeV -> Eikonal amplitude = 42686274.789478235j
√s = 788 GeV -> Eikonal amplitude = 56187351.67775192j
√s = 885 GeV -> Eikonal amplitude = 71651131.89210066j
√s = 981 GeV -> Eikonal amplitude = 89102517.91616419j
√s = 1077 GeV -> Eikonal amplitude = 108564039.65072395j
√s = 1173 GeV -> Eikonal amplitude = 130056282.92470346j
√s = 1269 GeV -> Eikonal amplitude = 153598210.19343966j
√s = 1365 GeV -> Eikonal amplitude = 179207406.85581127j
√s = 1462 GeV -> Eikonal amplitude = 206900274.979605j
√s = 1558 GeV -> Eikonal amplitude = 236692188.08241618j
√s = 1654 GeV -> Eikonal amplitude = 268597617.34100336j
√s = 1750 GeV -> Eikonal amplitude = 302630235.3413661j
√s = 1846 GeV -> Eikonal amplitude = 338803002.8426305j
√s = 1942 GeV -> Eikonal amplitude = 377128242.01828647j
√s = 2038 GeV -> Eikonal amplitude = 417617698.7155962j
√s = 2135 GeV -> Eikonal amplitude = 46028259

In [12]:
# lst_chi_imag = [chi.imag for chi in lst_chi]
lst_amp_eik_imag = [amp.imag for amp in lst_amp_eik]
lst_born_amp_imag = [born_amp.imag for born_amp in lst_born_amp]
print(lst_born_amp)
print(lst_born_amp_imag)
print(len(lst_sqrt_s), len(lst_born_amp_imag))


[78.41389592681145j, 1490109.4873411304j, 6423638.869562708j, 15139899.987069093j, 27837066.141512733j, 44660155.46798415j, 65724752.28819218j, 91127489.3288743j, 120951688.08989424j, 155270767.32100576j, 194150469.84636205j, 237650401.90699938j, 285825143.95655876j, 338725079.7989731j, 396397032.6674698j, 458884764.3689249j, 526229374.5007638j, 598469624.9733326j, 675642207.5383229j, 757781967.0470047j, 844922089.7807616j, 937094263.8426802j, 1034328816.9251059j, 1136654835.5549896j, 1244100269.0246596j, 1356692020.546476j, 1474456027.6620028j, 1597417333.5462039j, 1725600150.5439067j, 1859027917.0376782j, 1997723348.5574195j, 2141708483.8909173j, 2291004726.8328023j, 2445632884.1102986j, 2605613199.943152j, 2770965387.628281j, 2941708658.4844136j, 3117861748.445721j, 3299442942.5547743j, 3486470097.5725217j, 3678960662.895318j, 3876931699.9455247j, 4080399900.1821365j, 4289381601.86055j, 4503892805.65588j, 4723949189.251216j, 4949566120.981221j, 5180758672.611655j, 5417541631.327026j

In [29]:
lst_sigma_tot_eik = []
for sqrt_s_val, amp_val in zip(lst_sqrt_s_eik, lst_amp_eik_imag):
    s_val = sqrt_s_val**2 
    sigma_tot_val = ((4*np.pi)/s_val) * amp_val * 0.389379323
    print(f's = {s_val}, amp eik = {amp_val}, sigma tot eik = {sigma_tot_val}')
    lst_sigma_tot_eik.append(sigma_tot_val)

s = 250000.0, amp eik = 21456947.609921902, sigma tot eik = 419.96266388905224
s = 355399.4082840237, amp eik = 31120034.390653092, sigma tot eik = 428.455890096907
s = 479289.94082840247, amp eik = 42686274.789478235, sigma tot eik = 435.78541518475504
s = 621671.5976331361, amp eik = 56187351.67775192, sigma tot eik = 442.2423062633969
s = 782544.3786982249, amp eik = 71651131.89210066, sigma tot eik = 448.0194092638208
s = 961908.2840236687, amp eik = 89102517.91616419, sigma tot eik = 453.25130330971007
s = 1159763.3136094674, amp eik = 108564039.65072395, sigma tot eik = 458.0357517366847
s = 1376109.4674556213, amp eik = 130056282.92470346, sigma tot eik = 462.4460821527926
s = 1610946.7455621304, amp eik = 153598210.19343966, sigma tot eik = 466.5387495455987
s = 1864275.1479289944, amp eik = 179207406.85581127, sigma tot eik = 470.35817340156575
s = 2136094.6745562134, amp eik = 206900274.979605, sigma tot eik = 473.9399520644352
s = 2426405.3254437875, amp eik = 236692188.0824

In [30]:

# add_total_trace(fig_sigma_tot, lst_sqrt_s, lst_sigma_tot_eik, color='red', label='sigma tot eik', line_style='solid')
# add_total_trace(fig_sigma_tot, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='sigma tot born', line_style='solid')


#-----------------------------------------------------------------------------------------------
#----


fig_sigma_tot = go.Figure()

# add_total_trace(fig_sigma_tot, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='sigma tot born', line_style='solid')
add_total_trace(fig_sigma_tot, lst_sqrt_s_eik, lst_sigma_tot_eik, color='red', label='sigma tot eik', line_style='solid')

# add_data_trace(fig_sigma_tot, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig_sigma_tot.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]'
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig_sigma_tot.update_xaxes(gridcolor='lightgray')
fig_sigma_tot.update_yaxes(gridcolor='lightgray')

fig_sigma_tot.show(renderer="browser")


In [ ]:
# import numpy as np
# from scipy.special import j0

# # --- setup parameters ---
# step = 200
# lst_amp_eik_dif = []

# lst_q_integration = np.linspace(0, 0.2, 50)
# lst_b_integration = np.linspace(0, 30, 50)
# # lst_sqrt_s = np.arange(500, 13000, step)
# lst_sqrt_s = np.linspace(500, 13000, 131)
# lst_q_diff_eik = np.linspace(0, 0.1, 50)

# # compute step sizes for the Riemann sum
# dq = lst_q_integration[1] - lst_q_integration[0]
# db = lst_b_integration[1] - lst_b_integration[0]

# for q_diff_eik_val in lst_q_diff_eik:

#     sqrt_s = 13000
#     s = sqrt_s**2

#     # initialize amplitude for this sqrt(s)
#     eik_amp_sum = 0

#     # --- integration over b ---
#     for b_val in lst_b_integration:
#         chi_sum = 0

#         # --- integration over q ---
#         for q_val in lst_q_integration:
#             # full_int must be your differential amplitude function
#             diff_t = full_int(
#                 ensemble_parameters['atlas']['pl']['mg'],
#                 ensemble_parameters['atlas']['pl']['a1'],
#                 ensemble_parameters['atlas']['pl']['a2'],
#                 m2_pl,
#                 q_val,
#                 sqrt_s
#             )

#             t = -q_val
#             born_amp = amp_calculation(
#                 diff_t,
#                 s,
#                 ensemble_parameters['atlas']['pl']['epsilon'],
#                 t
#             )

#             chi_val = (1 / s) * (q_val * j0(b_val * q_val)) * born_amp

#             # Riemann sum contribution (Δq)
#             chi_sum += chi_val * dq

#         # after q integration, apply factor and accumulate over b (Δb)
#         factor = 1 - np.exp(1j * chi_sum)
#         eik_amp_value = (1j * s) * b_val * j0(b_val * q_diff_eik_val) * factor
#         eik_amp_sum += eik_amp_value * db

#     lst_amp_eik_dif.append(eik_amp_sum)
#     print(f"√s = {q_diff_eik_val} GeV -> Eikonal amplitude = {eik_amp_sum}")



√s = 0.0 GeV -> Eikonal amplitude = 20750233128.348415j
√s = 0.0020408163265306124 GeV -> Eikonal amplitude = 20745449599.269802j
√s = 0.004081632653061225 GeV -> Eikonal amplitude = 20731104373.481716j
√s = 0.006122448979591837 GeV -> Eikonal amplitude = 20707213526.77183j
√s = 0.00816326530612245 GeV -> Eikonal amplitude = 20673803823.617794j
√s = 0.010204081632653062 GeV -> Eikonal amplitude = 20630912674.493176j
√s = 0.012244897959183675 GeV -> Eikonal amplitude = 20578588076.213364j
√s = 0.014285714285714287 GeV -> Eikonal amplitude = 20516888535.438164j
√s = 0.0163265306122449 GeV -> Eikonal amplitude = 20445882975.481422j
√s = 0.018367346938775512 GeV -> Eikonal amplitude = 20365650626.610268j
√s = 0.020408163265306124 GeV -> Eikonal amplitude = 20276280900.048874j
√s = 0.022448979591836737 GeV -> Eikonal amplitude = 20177873245.933693j
√s = 0.02448979591836735 GeV -> Eikonal amplitude = 20070536995.497807j
√s = 0.02653061224489796 GeV -> Eikonal amplitude = 19954391187.793045j


In [ ]:
# lst_amp_eik_diff_imag = [amp.imag for amp in lst_amp_eik_dif]
# lst_born_amp_diff_imag = [born_amp.imag for born_amp in lst_amp_born_diff]

In [ ]:

# fig_amps = go.Figure()

# add_total_trace(fig_amps, lst_q_diff_eik, lst_amp_eik_diff_imag, color='red', label='diff amp eik', line_style='solid')
# add_total_trace(fig_amps, lst_q_diff_eik, lst_born_amp_diff_imag, color='blue', label='diff amp born', line_style='solid')
# #-----------------------------------------------------------------------------------------------
# #----

# fig_amps.update_layout(
#     title='Amp diff',
#     xaxis_title='|t| (GeV²)',
#     yaxis_title='dσ/dt (mb/GeV²)',
#     yaxis_type='log',
#     legend_title='Mass Model',
#     plot_bgcolor='white',
#     hovermode='x unified'
# )

# fig_amps.update_xaxes(gridcolor='lightgray')
# fig_amps.update_yaxes(gridcolor='lightgray')


# fig_amps.show(renderer="browser")